# 01 — Simulate + compare (TWAS-structured GReX)

Runs the full pipeline on **realistic TWAS GReX** (`simulate_twas_dataset`): cis-genotypes with
LD → sparse cis-eQTL effects → predicted expression (correlated across genes) → trait from a
sparse causal-gene set. This is the setting where the **Bayesian shrinkage** arm can beat the
**elastic-net** baseline. (For a pure wiring smoke on independent genes, use `simulate_dataset`.)

## Setup

In [ ]:
import numpy as np
import ptgs_bc as ptgs
from ptgs_bc import (simulate_twas_dataset, simulate_dataset, ElasticNetBuilder, BayesBuilder,
                     run_benchmark, summary_table, per_fold_table, plot_performance)
%matplotlib inline
print("ptgs_bc", ptgs.__version__)

## Simulate TWAS GReX — p ≫ n, sparse + correlated

Shrinkage priors are expected to help when there are many genes, few samples, and a sparse set
of true effects among correlated features.

In [ ]:
ds, true_w = simulate_twas_dataset(
    n_samples=180, n_genes=250, n_snps_per_gene=15, window_overlap=8, ld_rho=0.6,
    eqtl_sparsity=0.3, n_causal_genes=8, trait_pve=0.4, effect_sd=1.5, seed=0)
C = np.corrcoef(ds.grex.to_numpy().T)
print(f"p={ds.n_genes} genes, n={ds.n_samples} samples, 8 causal | "
      f"adjacent-gene |corr|={np.abs(np.diag(C,1)).mean():.3f}")

## Elastic-net vs. Bayesian through the same nested CV

In [ ]:
res = run_benchmark(
    [ElasticNetBuilder(), BayesBuilder(p0=8, num_warmup=300, num_samples=300)],
    ds, outer_k=5, seed=0)
summary_table(res)

In [ ]:
per_fold_table(res).pivot(index="fold", columns="builder", values="value").round(4)

## Performance comparison figure

Left: mean ± sd per arm with fold dots. Right: per-fold Bayesian (y) vs elastic-net (x) with a
y=x line — points above the line are folds where the Bayesian arm wins.

In [ ]:
fig = plot_performance(res)
fig

## Reading the result

In this p ≫ n, sparse, correlated regime the Bayesian (regularized-horseshoe) arm typically
shows a **higher mean and lower variance** than elastic net. The advantage shrinks (or reverses)
when p < n or the signal is dense — try `n_genes=100, n_samples=300` to see elastic net catch up.
The margin also grows with sparser/stronger signal, a better-calibrated `p0`, and more MCMC draws.